# EDA on ultimate marathon data to explore deeper what should be cleaned

In [0]:
df_um = spark.sql("SELECT * FROM marathos.bronze.raw_ultra_marathon")

display(df_um.limit(5))

Databricks visualization. Run in Databricks to view.

In [0]:
# saw club name with unsupported letters, investigate more

df_club_athletes = df_um.groupBy("Athlete club") \
                        .count() \
                        .withColumnRenamed("count", "amount") \
                        .orderBy("count", ascending = True)
                                            
df_club_athletes.limit(5).display()

In [0]:
%sql
-- see if an athlete have been in different clubs

SELECT DISTINCT
    `Athlete ID`,
    `Athlete club`
FROM 
    marathos.bronze.raw_ultra_marathon
ORDER BY `Athlete ID`
LIMIT 5;


## Check similarities and differences

In [0]:
%sql

-- See if any gender and age category mismatch

SELECT
    `Athlete gender`,
    `Athlete age category`,
    COUNT(*) as amount_rows
FROM 
    marathos.bronze.raw_ultra_marathon
WHERE 
    (`Athlete gender` = 'M' AND LEFT(`Athlete age category`, 1) != 'M')
    OR
    (`Athlete gender` = 'F' AND LEFT(`Athlete age category`, 1) != 'W')
GROUP BY
    `Athlete gender`,
    `Athlete age category`;

In [0]:
%sql
-- See if year of event and event dates matches
-- Collect year from first date if multiple

SELECT
    `Year of event`,
    SUBSTRING(`Event dates`, 7, 4) AS extract_start_year,
    `Event dates`,
    COUNT(*) as amount_rows
FROM 
    marathos.bronze.raw_ultra_marathon
WHERE 
    (TRY_CAST(SUBSTRING(`Event dates`, 7, 4) AS BIGINT) != `Year of event`)
GROUP BY
    `Year of event`,
    `Event dates`;

In [0]:
df_country = spark.sql("SELECT * FROM marathos.bronze.raw_country_reference")

df_country.limit(5).display()

In [0]:
%sql
-- Same code with matches to see it works

SELECT
    `Year of event`,
    SUBSTRING(`Event dates`, 7, 4) AS extract_start_year,
    `Event dates`
FROM 
    marathos.bronze.raw_ultra_marathon
WHERE 
    (TRY_CAST(SUBSTRING(`Event dates`, 7, 4) AS BIGINT) != `Year of event`)
GROUP BY
    `Year of event`,
    `Event dates`
LIMIT 5;

In [0]:
%sql
-- compare Event distance/length and Athlete performance
-- check if km or mi = h

SELECT
  `Event distance/length`,
  `Athlete performance`
FROM
  marathos.bronze.raw_ultra_marathon
WHERE
  (
    (
      RIGHT(`Event distance/length`, 2) = 'km'
      OR RIGHT(`Event distance/length`, 2) = 'mi'
    )
    AND RIGHT(`Athlete performance`, 1) != 'h'
  )
  OR (
    RIGHT(`Event distance/length`, 1) = 'h'
    AND (RIGHT(`Athlete performance`, 2) != 'km')
  )
  OR (
    RIGHT(`Athlete performance`, 2) = 'km'
    AND RIGHT(`Event distance/length`, 1) != 'h'
  )
  OR (
    RIGHT(`Athlete performance`, 1) = 'h'
    AND (
      RIGHT(`Event distance/length`, 2) != 'km'
      AND RIGHT(`Event distance/length`, 2) != 'mi'
    )
  )
GROUP BY
  `Event distance/length`,
  `Athlete performance`
LIMIT 5;

In [0]:
%sql

SELECT
    `Event dates`,
    `Event name`,
    `Event distance/length`,
    `Athlete performance`
FROM
    marathos.bronze.raw_ultra_marathon
WHERE `Event distance/length` ILIKE "%etapp%"
GROUP BY 
    `Event dates`,
    `Event name`,
    `Event distance/length`,
    `Athlete performance`
ORDER BY 
    `Event name`, `Event distance/length`
LIMIT 5;

In [0]:
%sql
-- count age from year of event and year of birth
-- check how many are under 18 and over 80

SELECT
  `Year of event`,
  `Athlete year of birth`,
  (`Year of event` - `Athlete year of birth`) AS athlete_age
FROM
  marathos.bronze.raw_ultra_marathon
WHERE
  (`year of event` - `Athlete year of birth`) < 18
  OR (`year of event` - `Athlete year of birth`) > 80
GROUP BY 
    `Year of event`,
    `Athlete year of birth`
  LIMIT 5;

### Summary of things to clean 

#### Country
- break out country from "Event name"
- join with country code tables -> get right match -> switch to country
- for country codes without match -> "unknown" or "missing"?

#### Extract/split 
- extract country from Event name
- split performace between h and km
- split distance/length between digits and kind (km or h) for possibility to calculate

#### Convert
- mi to km in Event distance/length

#### Calculate
- km/h and replace athlete avreage speed

#### Nulls
- Athlete performance -> remove row
- Athlete club - "none/unknown"
- Athlete country -> "unknown"
- Athlete year of birth - -> "Unknown"
- Athlete gender -> "unknown"
- Athlete age category -> "unknown
- Athlete average speed -> calculate, if missing -> remove row

#### Data types
- Event dates -> date
- Athlete average speed -> float/double

#### Rename columns
- convert to snake_case and lower case

#### Remove
- rows with d (days) as length/time
- rows where age (year of event - year of birth) is less than 10 and more then 90,
- rows where Athlete gender and Athlete age group (gender) does not match
- rows with etapp (all rows have 'event date' dd.-dd.mm.yy, the different etaps have different names, hard to split distance/length and performans over several dates)

#### Create ID
- event id
- date id
- result id

#### Other insights
- change from UTF-8 -> latin1 for more supported letters

# Cleaning

## Remove unwanted rows

## Standardize column names

## Extract/Split values

## Remove rows due to logic

## Change datatypes

## Standardize values

## Calculation

## Join tables to obt (one big table)

## Create IDs